In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime

def options_chain(symbol):

    tk = yf.Ticker(symbol)
    # Expiration dates
    exps = tk.options
    print(exps)

    # Get options for each expiration
    options = pd.DataFrame()
    for e in exps:
        opt = tk.option_chain(e)
        opt = pd.DataFrame().append(opt.calls).append(opt.puts)
        opt['expirationDate'] = e
        options = options.append(opt, ignore_index=True)

    # Bizarre error in yfinance that gives the wrong expiration date
    # Add 1 day to get the correct expiration date
    options['expirationDate'] = pd.to_datetime(options['expirationDate']) + datetime.timedelta(days = 1)
    options['dte'] = (options['expirationDate'] - datetime.datetime.today()).dt.days / 365
    
    # Boolean column if the option is a CALL
    options['CALL'] = options['contractSymbol'].str[4:].apply(
        lambda x: "C" in x)
    
    options[['bid', 'ask', 'strike']] = options[['bid', 'ask', 'strike']].apply(pd.to_numeric)
    options['mark'] = (options['bid'] + options['ask']) / 2 # Calculate the midpoint of the bid-ask
    
    # Drop unnecessary and meaningless columns
    options = options.drop(columns = ['contractSize', 'currency', 'change', 'percentChange', 'lastTradeDate', 'lastPrice'])

    return options

In [ ]:
df = options_chain("^spx")

In [ ]:
df

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import scipy.stats as si

In [ ]:
df

In [ ]:
df.strike.values

In [ ]:
df["impliedVolatility"][1]

In [ ]:
S = df.strike.values.astype(np.float64)
T = df.dte.values.astype(np.float64)
ImpliedVolatility = np.zeros((len(T),len(S)))

for j in range(len(S)):
    for i in range(len(T)):
        ImpliedVolatility[i,j] = df["impliedVolatility"][j]

In [ ]:
T

In [ ]:
ImpliedVolatility=ImpliedVolatility.astype(np. float64)

In [ ]:
ImpliedVolatility.plot()

In [ ]:
fig = plt.figure(figsize= (15, 8))
ax = fig.gca(projection='3d')
S, T = np.meshgrid(S, T)
surf = ax.plot_surface(S, T, ImpliedVolatility, rstride=2, cstride=2, cmap=plt.cm.coolwarm, linewidth=0.5, antialiased=True)
ax.set_xlabel('Stock Price')
ax.set_ylabel('Time to Expiry')
ax.set_zlabel('ImpliedVolatility')
fig.colorbar(surf, shrink=0.5, aspect=5);

In [ ]:
st = S
ttm=T
a2=ImpliedVolatility
fig = plt.figure(figsize= (15, 15))
ax = fig.gca(projection='3d')
surf = ax.plot_surface(st, ttm, a2, rstride=2, cstride=2, cmap=plt.cm.coolwarm, linewidth=0.1, antialiased=True)
ax.set_xlabel('Strike')
ax.set_ylabel('Time to Maturitre')
ax.set_zlabel('Volatile Implied')
fig.colorbar(surf, shrink=0.5, aspect=1,);